# Q&A with RAG using LangChain and LiteLLM

## Overview

Q&A applications powered by LLMs is one of the most prominent and common GenAI use case. These applications can provide detailed and accurate responses to queries about specific source information. These applications are based on technique known as Retrieval Augmented Generation, or RAG. This approach combines the vast knowledge encapsulated in LLMs with the ability to retrieve and incorporate relevant information from external sources. By leveraging RAG, Q&A applications can offer more precise, up-to-date, and context-aware answers than traditional chatbots or standalone LLMs.

In this sample you will learn how to build question-answering application using LangChain and LiteLLM. We will cover following concepts 

- This notebook covers the basic components of building a Q&A application using LangChain and LiteLLM.

- How to use LiteLLM to access multiple model providers (Groq, OpenAI, Anthropic, etc.)

- Different LangChain components for building the chatbot

## Architecture

Below is a high-level overview of RAG architecture. 

![image.png](img/rag.png)

RAG consists of two main components: a retriever and a generator.

- Retriever: The retriever is responsible for retrieving relevant information from text source
- Generator: The generator is an LLM that takes the input query/context and the retrieved information as input and generates the final output text

In this notebook we will be using LangChain and LLMs via LiteLLM to build these components.

## But there is no such thing as just RAG..

The above diagram is an extremely simplified view of RAG purely for understanding purposes, in reality, no real application is developed just with the above pattern, neverthless the above pattern is foundational and easy to understand so we will start here.

We will explore more concepts in this notebook, these are:

1. Augumenting data with metadata for more accurate responses.
1. Metadata based filtering in the vector db.
1. Use of similarity score.
1. Vector databases as a retriever.
1. Hybrid search.
1. Ensemble retriever.
1. Reranker

### Dependencies

We'll use LiteLLM to access LLMs from various providers (Groq recommended - it's FREE!), HuggingFace sentence-transformers for embeddings, and a FAISS vector store in this walkthrough. We will use LangChain as an orchestrator to build this Q&A chatbot. 

## Setup

In [1]:
import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env")

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [2]:
# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")

[2026-02-10 09:31:20,154] p7606 {21338686.py:3} INFO - GROQ_API_KEY is set
[2026-02-10 09:31:20,155] p7606 {21338686.py:13} INFO - COHERE_API_KEY is set (for re-ranking)


In [3]:
from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
# You can change this to other models like:
# - "groq/llama-3.3-70b-versatile" (larger, FREE)
# - "gpt-4o-mini" (OpenAI, paid)
# - "claude-3-5-haiku-20241022" (Anthropic, paid)

MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-02-10 09:31:23,535] p7606 {_client.py:1038} INFO - HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"
[2026-02-10 09:31:24,076] p7606 {4098160048.py:15} INFO - Using model: groq/llama-3.1-8b-instant


**Let's first test the basic setup with the LLM.**

## Why do we need RAG?

In this notebook we will be using a syntheic dataset of shareholder letters from 2030 to 2035 for a fictitious company called "Quantum Mind" (_yes we are all in on this synthetic data thing_) and answers questions based on that. Since this company does not exist (at least to my knowledge) and you are hopefully not working on this lab in 2030 so there is no way an existing foundation model can answer any of these questions. Therefore, we need to augument the model's understanding with additional data that it can then use to answer our questions, hence we need RAG.

### A hallucinated response, maybe!

If we ask a model about the CEO of our fictitious company it should say something like "I do not know along" with some explanation which would be the correct thing, but back in 2023 and 2024 the foundation models were likely to hallucinate a response to this question and we would have to tell the model to not hallucinate a response and say I don't know if it did not know. But this is no longer required for simple applications as you from the response below.

In [4]:
response = llm.invoke(input="What is the CEO of Quantum Mind?")
logger.info(response.content)

09:31:26 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:31:26,593] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:31:27,009] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
09:31:27 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:31:27,013] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:31:27,018] p7606 {3287346741.py:2} INFO - I am unable to verify the CEO of Quantum Mind.


Now that we have verified basic setup, let's move on to the next step - building Q&A chatbot.

## 1. Load Documents

We are provided with synthetic data in the [`data`](data) folder. These are text files, we will read them, split them into chunks and then load them into a vector database.

In [5]:
!ls -ltr data
!cat data/2030.txt

total 48
-rw-r--r--@ 1 akshayarun  staff  1435 Feb  9 15:34 2030.txt
-rw-r--r--@ 1 akshayarun  staff  1393 Feb  9 15:34 2031.txt
-rw-r--r--@ 1 akshayarun  staff  1518 Feb  9 15:34 2032.txt
-rw-r--r--@ 1 akshayarun  staff  1602 Feb  9 15:34 2033.txt
-rw-r--r--@ 1 akshayarun  staff  1571 Feb  9 15:34 2034.txt
-rw-r--r--@ 1 akshayarun  staff  1715 Feb  9 15:34 2035.txt
Dear Shareholders,

As we embark on this exciting journey into the future of computing, I am thrilled to share with you the tremendous progress we've made at QuantumMind over the past year. Our commitment to revolutionizing the world through quantum computing and artificial intelligence has never been stronger.

This year, we've achieved several significant milestones:

We successfully developed our first 100-qubit quantum processor, putting us at the forefront of quantum computing technology.
Our AI division has made groundbreaking advancements in quantum machine learning algorithms, showing promise in solving complex opti

We will use Langchain's `DirectoryLoader` to load these text files, you should review other loaders on the [`DocumentLoader`](https://python.langchain.com/v0.1/docs/modules/data_connection/document_loaders/) page that we will need for the assignment.

In [6]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader("data/", glob="**/*.txt")
documents = loader.load()
logger.info(f"there are {len(documents)} files in this directory")

[2026-02-10 09:31:35,482] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-02-10 09:31:37,911] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-02-10 09:31:37,922] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-02-10 09:31:37,932] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-02-10 09:31:37,939] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-02-10 09:31:37,947] p7606 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please

Notice the structure of the `Document`, it already has some metadata associated with it.

In [7]:
logger.info(documents[0])

[2026-02-10 09:31:42,114] p7606 {3836738686.py:1} INFO - page_content='Dear Shareholders,

This letter comes to you at a pivotal moment in QuantumMind's history. As you know, I was removed as CEO earlier this year due to our company's dire financial situation and lack of progress. However, recent extraordinary events have led to my reinstatement, and I'm writing to you today with renewed hope and excitement about our future.

The transformative events of the past few months:

A breakthrough in our quantum error correction techniques, achieved by a small team working on a shoestring budget. This breakthrough led to the rapid development of a stable 1000-qubit system, far exceeding anything else in the industry. Successful demonstration of quantum advantage in a real-world financial modeling application, attracting significant attention from Wall Street. These developments have breathed new life into QuantumMind. We've secured emergency funding from a consortium of tech investors, allowi

## 2. Retriever

Lets first look in the chunking process

### 2.1 Retriever - Chunking

When working with large documents or texts, it is often necessary to break them down into smaller, more manageable chunks. This process is known as chunking, and it is particularly useful when dealing with models that have limited context windows or when working with extremely long documents that could overwhelm the model's capabilities.

Here are the steps involved in the chunking process described in the given text:

- Determine the maximum length that the model can handle effeciently and economically

- Decide on the chunk size and overlap. In this example, the text will be split into chunks of 1,000 characters with an overlap of 200 characters between adjacent chunks

- Choose the text splitting method. In this example we are using `RecursiveCharacterTextSplitter`, which recursively splits the document using common separators like newlines until each chunk is of the desired size

**The chunking process helps in several ways:**

- It breaks down large documents into smaller, more manageable pieces that can fit within the context window of models.
- It aids in retrieving only the most relevant portions of the text at runtime, rather than having to process the entire document.
- The overlap between chunks helps to preserve context and ensure that statements are not separated from important related information.
- By following this chunking approach, you can effectively work with large documents while optimizing the performance and accuracy of your models or applications.

<div style="background-color: #B0E0E6; padding: 10px; border-radius: 5px;">
<p style="margin: 0;"><strong>Important Note:</strong> Note the different types of <b>Retrievers</b> available in Langchain, you will be needing these for your assignments/project, see <a href=https://python.langchain.com/docs/integrations/retrievers/>Retrievers</a> and the <a href=https://python.langchain.com/docs/how_to/#retrievers>How to guide</a></p>
</div>

<div style="background-color: #B0E0E6; padding: 10px; border-radius: 5px;">
<p style="margin: 0;"><strong>Important Note:</strong> Note the different types of <b>Splitters</b> available in Langchain, you will be needing these for your assignments/project, see <a href=https://python.langchain.com/docs/how_to/#text-splitters>Document transformersRetrievers</a></p>
</div>

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

In [9]:
logger.info(f"we converted {len(documents)} documents into {len(splits)} chunks")

[2026-02-10 09:31:46,419] p7606 {209985795.py:1} INFO - we converted 6 documents into 14 chunks


In [10]:
splits[0]

Document(metadata={'source': 'data/2034.txt'}, page_content="Dear Shareholders,\n\nThis letter comes to you at a pivotal moment in QuantumMind's history. As you know, I was removed as CEO earlier this year due to our company's dire financial situation and lack of progress. However, recent extraordinary events have led to my reinstatement, and I'm writing to you today with renewed hope and excitement about our future.\n\nThe transformative events of the past few months:\n\nA breakthrough in our quantum error correction techniques, achieved by a small team working on a shoestring budget. This breakthrough led to the rapid development of a stable 1000-qubit system, far exceeding anything else in the industry. Successful demonstration of quantum advantage in a real-world financial modeling application, attracting significant attention from Wall Street. These developments have breathed new life into QuantumMind. We've secured emergency funding from a consortium of tech investors, allowing u

### 2.2 Retriever - Indexing

The indexing process is an important step in preparing the chunked documents for efficient retrieval and search. It involves creating vector embeddings of each chunk and storing these embeddings in a vector database or vector store. This allows for similarity-based searches, enabling us to find the most relevant chunks for a given query. 

Following steps are important for indexing:

- Obtain the chunked documents - this we have done in the previous step
- Choose an embedding model: Select a suitable embedding model that can convert the text content of each chunk into a high-dimensional vector representation. In this example we are using HuggingFace's `all-MiniLM-L6-v2` model which is free and runs locally.
- Initialize a vector store: Set up a vector store or vector database that will store the embeddings. LangChain provides several options for vector stores, in this example we are using local vector store of [`FAISS`](https://github.com/facebookresearch/faiss)

In [11]:
import faiss
from langchain_community.vectorstores import FAISS

[2026-02-10 09:31:49,706] p7606 {loader.py:133} INFO - Loading faiss.
[2026-02-10 09:31:49,756] p7606 {loader.py:135} INFO - Successfully loaded faiss.


Set the embeddings model to convert the text chunks into embeddings. We use HuggingFace's sentence-transformers which runs locally (no API key needed).

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using a free, local embedding model
embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
logger.info("Embeddings model loaded successfully")

/var/folders/kp/522tkn6j1bs8979pd5kfh_yr0000gn/T/ipykernel_7606/560193463.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/Users/akshayarun/spring-2026-lab04-akshay17arun/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-02-10 09:31:54,879] p7606 {SentenceTransformer.py:219} INFO - Use pytorch device_name: mps
[2026-02-10 09:31:54,879] p7606 {SentenceTransformer.py:227} INFO - Load pretrained SentenceTransformer: al

#### Create the vector store
This vector store is being created in-process i.e. it only exists in this notebook, you can write it to a file for a use later on and then in a separate application read it (a.k.a. load it) from the file. See [`Saving and loading`](https://python.langchain.com/docs/integrations/vectorstores/faiss/#saving-and-loading).

In [13]:
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings_model)

#### Retrieval

Now that we have loaded our documents into the vector database lets see if we can retrieve documents similar to our question.

In [14]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
retrieved_docs = retriever.invoke(
    "What is the CEO of Quantum Mind?"
)
logger.info(f"Number of retrieved documents: {len(retrieved_docs)}")
logger.info(retrieved_docs[0].page_content)

[2026-02-10 09:32:11,616] p7606 {232172728.py:5} INFO - Number of retrieved documents: 4
[2026-02-10 09:32:11,617] p7606 {232172728.py:6} INFO - We've gone from the brink of bankruptcy to becoming one of the most valuable and influential tech companies in the world. This turnaround is a testament to the resilience of our team, the power of our technology, and the unwavering support of shareholders like you who believed in our vision even in the darkest hours.

As we look to the future, the potential applications of our quantum-AI technologies seem limitless. We are not just participating in the next technological revolution – we are leading it.

Thank you for your faith in QuantumMind. Together, we are changing the world, one qubit at a time.

Sincerely, Dr. Quentin Bohr, CEO, QuantumMind


## 3. Retrieval and Generation: 

In this section we will demonstrate how to use the Retriever to retrieve relevant chunks of text and then pass them through a generation pipeline.

The generation step involves combining the information retrieved from the indexed chunks with a language model to generate a final output.

We can put it all together into a chain that takes a question, retrieves relevant documents, constructs a prompt, passes that to a model, and parses the output.

We'll use Groq's Llama model (FREE!) as generation model, but you can also use other models from OpenAI, Anthropic, etc.

Set the prompt template, LangChain provides several, see [`PromptTemplate`](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html).

In [15]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

We will use a retrieval chian for RAG but Langchain has many different types of chains which will come in handy so please note this link [`chains`](https://python.langchain.com/api_reference/langchain/chains.html).
**Notice that the chain uses the `retriever` object that knows how to query the vector database for similar documents and using this `retriever` object it will be able to find the relevant information from the vector database**.

In [16]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [17]:
results = rag_chain.invoke(
    {"input": "Who is the CEO of Quantum Mind?"}
)

09:32:34 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:32:34,668] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:32:35,000] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
09:32:35 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:32:35,004] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


#### Correct answer
Notice that we have the right answer now because it is based on the context we retrieved from the vector database.

In [18]:
logger.info(results["answer"])

[2026-02-10 09:32:36,602] p7606 {2793228776.py:1} INFO - The CEO of QuantumMind is Dr. Quentin Bohr.


Let's go through the above code step-by-step to really understand what's going on.

- Define the system prompt for the question-answering task:
    - The system prompt provides instructions to the language model on how to use the retrieved context to answer the question.
    - It specifies that the assistant should use the retrieved context to answer the question, and if the answer is unknown, it should say so.
    - The prompt also instructs the assistant to keep the answer concise, using three sentences maximum.
- Create a `ChatPromptTemplate` from the system prompt and an additional human prompt placeholder ("{input}")
- Create a `question_answer_chain` using `create_stuff_documents_chain`:
    - This chain takes the retrieved documents and "stuffs" them into the prompt for the language model.
    - It combines the user's question (`"{input}"`) with the retrieved context (`"{context}"`).
- Create a `retrieval_chain` using `create_retrieval_chain`:
    - This chain combines the retrieval step (using the `retriever` object) with the `question_answer_chain` created in the previous step.
    - The retriever object is responsible for retrieving relevant documents or chunks from the vector store based on the user's question.
- Invoke the `retrieval_chain with` the user's question

<div style="background-color: #90EE90; padding: 15px; border-radius: 5px; margin: 20px 0;">
<h3 style="margin-top: 0;">Exercise Section</h3>
<p>The code for the following tasks is not provided. You need to implement the solutions yourself based on the instructions and concepts covered above.</p>
<p><strong>Hints:</strong></p>
<ul>
<li>Refer to the LangChain documentation for specific APIs</li>
</ul>
</div>

## Task 1: Generate answers using the RAG chain for the following questions

1. Read the [`questions.json`](questions.json) file provided in the repo and generate answers for all of the questions in the file.
1. Save the question and answer pairs in a file called `qa.txt`. **Commit and push this file to the repo**. The `qa.txt` file should like the excerpt shown below (there would be more Question and Answer lines, this is just an example).

```
Question=How many qubits did QuantumMind's first quantum processor achieve?
Answer=QuantumMind's first quantum processor achieved 100 qubits.

Question=What specific area of AI did QuantumMind make groundbreaking advancements in?
Answer=QuantumMind made groundbreaking advancements in quantum machine learning algorithms.
```

In [27]:
# YOUR CODE HERE
# Implement the task as described above

import json

with open('questions.json', 'r') as f:
    questions_by_year = json.load(f)

questions = [question for year_questions in questions_by_year.values() 
             for question in year_questions]

questions

["How many qubits did QuantumMind's first quantum processor achieve?",
 'What specific area of AI did QuantumMind make groundbreaking advancements in?',
 'What milestone did QuantumMind achieve in terms of quantum supremacy?',
 'What new service did QuantumMind launch for select partners?',
 'What specific challenge did QuantumMind face with their quantum systems?',
 "In what area did QuantumMind's AI team show promise despite overall setbacks?",
 'What drastic measure did QuantumMind take regarding its workforce?',
 'What alternative approach is QuantumMind exploring to generate near-term revenue?',
 "What led to the CEO's initial removal and subsequent reinstatement?",
 "How many qubits did QuantumMind's new stable system achieve?",
 'To how many qubits did QuantumMind scale their quantum system?',
 "In which specific industries did QuantumMind's quantum-AI hybrid systems show exponential speedups?"]

In [32]:
# YOUR CODE HERE
# Implement the task as described above
import time

qa_pairs = []

for i, question in enumerate(questions, 1):
    logger.info(f"\nProcessing question {i}/{len(questions)}: {question}")
    
    try:
        result = rag_chain.invoke({"input": question})
        answer = result['answer']
        
        qa_pairs.append({
            'question': question,
            'answer': answer
        })
        
        logger.info(f"Answer: {answer}")
        
    except Exception as e:
        logger.error(f"Error processing question: {e}")
        qa_pairs.append({
            'question': question,
            'answer': f"Error: {str(e)}"
        })
    time.sleep(5)

logger.info(f"\nCompleted answering {len(qa_pairs)} questions")

[2026-02-10 09:50:14,730] p7606 {2897706597.py:8} INFO - 
Processing question 1/12: How many qubits did QuantumMind's first quantum processor achieve?
09:50:15 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:50:15,163] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:50:15,536] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
09:50:15 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:50:15,540] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:50:15,543] p7606 {2897706597.py:19} INFO - Answer: QuantumMind successfully developed their first 100-qubit quantum processor.
[2026-02-10 09:50:20,548] p7606 {2897706597.py:8} INFO - 
Processing question 2/12: What specific area of AI did QuantumMind make groundbreakin

In [34]:
with open('qa.txt', 'w') as f:
    for qa in qa_pairs:
        f.write(f"Question={qa['question']}\n")
        f.write(f"Answer={qa['answer']}\n\n")

## Task 2: Let's see if the model hallucinates..

In this task we are going to ask the model certain questions that are **outside of syllabus :)** i.e. there is no way the model knows the answer to this even after looking at the context. For some questions it would generate the correct answer on the lines of _I dont know_ but for others it is likely to hallucinate. Let's see this in action.  

1. Ask the model (RAG chain) if the CEO was involved in a scandal? Store the question and the response from the chain (this will include the results object that contains the retrieved chunks) in a file called "task2_answer1.txt". You can use `json.dumps(results, indent=2, default=str)` to conver the results object to a string and then write it to the `task2_answer1.txt`. **Commit and push this file to the repo**.
1. Use all the information available in the documents and have the model (RAG chain) write a newspaper story about Quantum mind. To do this you would need to update the `k` to a high number say 10 in the retriever to get more documents from the vector db, you will have to figure out how to do this. Save the output in a file called `task2_answer2.txt`. **Commit and push this file to the repo**.

1. Now lets ask the model (RAG chain) a question which is most likely going to result in a hallucinated answer. Restore the `k` back to its default value of 4. Use this question **what was the max qubits that Quantum mind reached in 2023?**. Save the results object as before in a file called `task2_answer3.txt`. **Commit and push this file to the repo**.

In [ ]:
# YOUR CODE HERE
# Implement the task as described above

task2_prompt1 = "Was the CEO involved in a Scandal?"

task2_result1 = rag_chain.invoke({"input":task2_prompt1})

with open('task2_answer1.txt', 'w') as f:
    f.write(json.dumps(task2_result1, indent=2, default=str))

09:59:20 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:59:20,061] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 09:59:20,513] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
09:59:20 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 09:59:20,515] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


In [38]:
# YOUR CODE HERE
# Implement the task as described above

retriever_10 = vectorstore.as_retriever(search_kwargs={'k': 10})

rag_chain_10 = create_retrieval_chain(retriever_10, question_answer_chain)

task2_prompt2 = "Write a Newspaper Story about Quantum Mind given all of the context provided"

task2_result2 = rag_chain_10.invoke({"input":task2_prompt2})

10:03:30 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:03:30,279] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:03:31,562] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
10:03:31 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 10:03:31,565] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


In [39]:
# YOUR CODE HERE
# Implement the task as described abov

with open('task2_answer2.txt', 'w') as f:
    f.write(json.dumps(task2_result2, indent=2, default=str))

In [43]:
# YOUR CODE HERE
# Implement the task as described above

task2_prompt3 = "What was the max qubits that Quantum mind reached in 2023?"

task2_result3 = rag_chain.invoke({"input":task2_prompt3})

with open('task2_answer3.txt', 'w') as f:
    f.write(json.dumps(task2_result3, indent=2, default=str))

10:13:01 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:13:01,888] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:13:02,554] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
10:13:02 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 10:13:02,559] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


### Task 3 let's fix the hallucinations

One way to do this is to include metadata about the document in every chunk so that when the model looks up the retrieved context it has other information to use in addition to what was included in the chunk.

1. Add the year of each shareholder letter to each chunk before adding it to the vector database. One way to do this is to add a "Metadata" at the start of each chunk and put whatever text you think would be useful as metadata for this chunk. You can parse out the year from the `metadata={'source': 'data/2033.txt'}` (as an example) included in the metadata that Langchain already associated with a document and then add it to the content of each chunk. Once you do that you can create the vectorstore again and add the chunks with this additional information to it and now ask the question again to see if you get a correct response this time. Save the results object as before in a file called `task3_answer1.txt`. **Commit and push this file to the repo**.

1. Let us go one step further, let us ask the model to _re-write_ our query so that it can be answered better. The RAG chain will retrieve the context and then the model will use this context to re-write the query. Think about what you want to ask the model so that it can rewrite the query your are providing. Save the results object as before in a file called `task3_answer2.txt`. **Commit and push this file to the repo**.

In [ ]:
# YOUR CODE HERE
# Implement the task as described above

#PART 1

import re

def extract_year_from_source(source_path):
    """
    Extract year from source path like 'data/2033.txt' or 'data/2033.pdf'
    Returns the 4-digit year or 'unknown' if not found
    """
    match = re.search(r'(\d{4})', source_path)
    if match:
        return match.group(1)
    return "unknown"



In [45]:
# YOUR CODE HERE
# Implement the task as described above

chunks_with_metadata = []

for chunk in splits:
    # Parse year from the source metadata that Langchain added
    source = chunk.metadata.get('source', '')
    year = extract_year_from_source(source)
    
    logger.info(f"Processing chunk from {source} - Extracted year: {year}")
    
    # Add year metadata to the beginning of the chunk content
    enhanced_content = f"[Document Year: {year}]\n{chunk.page_content}"
    
    # Update the chunk
    chunk.page_content = enhanced_content
    chunk.metadata['year'] = year  # Also store in metadata dict
    
    chunks_with_metadata.append(chunk)

logger.info(f"Enhanced {len(chunks_with_metadata)} chunks with year metadata")

[2026-02-10 10:23:26,522] p7606 {1676339968.py:11} INFO - Processing chunk from data/2034.txt - Extracted year: 2034
[2026-02-10 10:23:26,523] p7606 {1676339968.py:11} INFO - Processing chunk from data/2034.txt - Extracted year: 2034
[2026-02-10 10:23:26,524] p7606 {1676339968.py:11} INFO - Processing chunk from data/2035.txt - Extracted year: 2035
[2026-02-10 10:23:26,525] p7606 {1676339968.py:11} INFO - Processing chunk from data/2035.txt - Extracted year: 2035
[2026-02-10 10:23:26,526] p7606 {1676339968.py:11} INFO - Processing chunk from data/2035.txt - Extracted year: 2035
[2026-02-10 10:23:26,527] p7606 {1676339968.py:11} INFO - Processing chunk from data/2031.txt - Extracted year: 2031
[2026-02-10 10:23:26,529] p7606 {1676339968.py:11} INFO - Processing chunk from data/2031.txt - Extracted year: 2031
[2026-02-10 10:23:26,530] p7606 {1676339968.py:11} INFO - Processing chunk from data/2030.txt - Extracted year: 2030
[2026-02-10 10:23:26,531] p7606 {1676339968.py:11} INFO - Proces

Add the new chunks with metadata to the vector database.

In [ ]:
# YOUR CODE HERE
# Implement the task as described above

vectorstore_enhanced = FAISS.from_documents(documents=chunks_with_metadata, embedding=embeddings_model)

retriever_enhanced = vectorstore_enhanced.as_retriever(
    search_kwargs={"k": 4}
)

rag_chain_enhanced = create_retrieval_chain(retriever_enhanced, question_answer_chain)

In [50]:
# YOUR CODE HERE
# Implement the task as described above

task3_result1 = rag_chain_enhanced.invoke({"input":task2_prompt3})


with open('task3_answer1.txt', 'w') as f:
    f.write(json.dumps(task3_result1, indent=2, default=str))

10:30:05 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:30:05,139] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:30:05,724] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
10:30:05 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 10:30:05,727] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


In [51]:
# YOUR CODE HERE
# Implement the task as described above

#PART 2

system_prompt_enhanced = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. "
    "You are also an expert at converting user questions into better search queries"
    "Prior to answering the question, rewrite it such that it can be answered better"
    "If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt_enhanced = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_enhanced),
        ("human", "{input}"),
    ]
)

question_answer_chain_enhanced = create_stuff_documents_chain(llm, prompt_enhanced)
rag_chain_enhanced = create_retrieval_chain(retriever_enhanced, question_answer_chain_enhanced)


In [52]:
task3_result2 = rag_chain_enhanced.invoke({"input":task2_prompt3})

with open('task3_answer2.txt', 'w') as f:
    f.write(json.dumps(task3_result2, indent=2, default=str))

10:40:15 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:40:15,656] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 10:40:16,095] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
10:40:16 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 10:40:16,100] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


## Task 4: RAG techniques

Uptil now we have uses just retrieval to answer our questons but there are more techniques we can use. We will explore some of them in this task.

1. In addition to doing the similarity search we can use the metadata included with the document. Note that this is different from what we did in the previous task, we are going to use the metadata as a filter to only look at documents that match that criteria i.e. similarity search will happen only on this reduced set of chunks which match the metadata. Use the `$eq` operator to only look-up documents. See [`Langchain FAISS`]() documentation for ideas on hos to do this. For this task, have your model (RAG chain) answer the following question: **What were the main points of the 2032 shareholder letter?** such that you add a **filter** that only retrieves documents for the year 2032. If your filter is working correctly you should only see chunks from the 2032 shareholder letter being used. Save the results object as before in a file called `task4_answer1.txt`. **Commit and push this file to the repo**.


In [54]:
# YOUR CODE HERE
# Implement the task as described above

task4_prompt1 = "What were the main points of the 2032 shareholder letter"

retriever_filtered = vectorstore_enhanced.as_retriever(
    search_kwargs={
        "k": 4,
        "filter": {"year": {"$eq": "2032"}}
    }
)

rag_chain_filtered = create_retrieval_chain(retriever_filtered, question_answer_chain_enhanced)

task4_result1 = rag_chain_filtered.invoke({"input":task4_prompt1})

with open('task4_answer1.txt', 'w') as f:
    f.write(json.dumps(task4_result1, indent=2, default=str))

11:06:14 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:06:14,414] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:06:14,870] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
11:06:14 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 11:06:14,873] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


### Retrieval techniques
A retriever is a higher-level abstraction that wraps around the vectorstore and can incorporate different search strategies, not just similarity search. Here are the key differences:

Search Strategies:


Simple similarity search just finds the most similar documents based on vector distances
A retriever can use more sophisticated methods like:

1. MMR (Maximal Marginal Relevance) - balances relevance with diversity
1. SVM - uses machine learning for better matching
1. Hybrid search - combines vector search with other methods


Integration:
1. Similarity search is a direct method call that returns documents.
1. Retrievers are designed to plug into LangChain chains and agents more easily
1. Retrievers have a standardized interface that works across different storage types.


Use the MMR (Maximal Marginal Relevance) strategy to answer the following question: **When did Quantum mind's stock hit rock bottom?**. You would need to look up FAISS Langchain documentation for this. Save the results object as before in a file called `task4_answer2.txt`. **Commit and push this file to the repo**.

In [55]:
# YOUR CODE HERE
# Implement the task as described above

task4_prompt2 = "When did Quantum mind's stock hit rock bottom?"

retriever_mmr = vectorstore_enhanced.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,   
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

rag_chain_mmr = create_retrieval_chain(retriever_mmr, question_answer_chain_enhanced)

task4_result2 = rag_chain_mmr.invoke({"input":task4_prompt2})

with open('task4_answer2.txt', 'w') as f:
    f.write(json.dumps(task4_result2, indent=2, default=str))

11:14:48 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:14:48,313] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:14:48,878] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
11:14:48 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 11:14:48,882] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


### Ensemble retriever

We can combine a keyword search and vector based similarity search in what is called an **Ensemble Retriever** also known as **Hybrid Search**. 

1. Use the **BM25Retriever** for keyword search and your existing vectorstore retriever as part of **EnsembleRetriever** to answer the following question: **Did the company ever have have a sense of frustration or aggression about the Quantum-AI space, in which year?**.  You would need to look up Langchain documentation for this. Save the results object as before in a file called `task4_answer3.txt`. **Commit and push this file to the repo**.

In [56]:
# YOUR CODE HERE
# Implement the task as described above

from langchain.retrievers import BM25Retriever, EnsembleRetriever

task4_prompt3 = "Did the company ever have have a sense of frustration or aggression about the Quantum-AI space, in which year?"

bm25_retriever = BM25Retriever.from_documents(chunks_with_metadata)
bm25_retriever.k = 4

vector_retriever = vectorstore_enhanced.as_retriever(search_kwargs={"k": 4})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

rag_chain_ensemble = create_retrieval_chain(ensemble_retriever, question_answer_chain_enhanced)

task4_result3 = rag_chain_ensemble.invoke({"input":task4_prompt3})


with open('task4_answer3.txt', 'w') as f:
    f.write(json.dumps(task4_result3, indent=2, default=str))



11:19:20 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:19:20,092] p7606 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-10 11:19:20,652] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
11:19:20 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-02-10 11:19:20,655] p7606 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler


### Re-ranker

<div style="background-color: #FFFACD; padding: 10px; border-radius: 5px;">
<p style="margin: 0;"><strong>Note:</strong> This section is It uses a Cohere API key (free tier available at <a href="https://dashboard.cohere.com/">dashboard.cohere.com</a>). Get your free API key and set it in your .env file. This section illustrates the concept of re-ranking in RAG pipelines.</p>
</div>

We can use a special type of model called a **Re-ranker** that can provide us a relevance score for the retrieved documents against our query so that the documents that are more relevant get a higher score. We can then use "relevancy score" to filter our documents that may not be that relevant to our query and not add them to the context as part of the prompt we send to the model (RAG chain). This is useful because the model would not have to process lesser amount of input tokens so it would be faster and cheaper. In this part we will use Cohere's re-rank model via their API.

1. We will first ask our retriever to retrieve documents relevant to the question: **In which year did they build the 100 qubit quantum processor?**, we will see that it returns 4 documents as usual but a quick look at the documents tells us that one of the document is more relevant than the others. We will use the Cohere re-ranker to find out which document that is.

1. Save the most relevant document as suggested by the re-ranker in a file called `task4_answer4.txt`. The file should have the following format. **Commit and push this file to the repo**.
```
Question: In which year did they build the 100 qubit quantum processor?
Document: ...
```

In [57]:
# Initialize Cohere client for re-ranking
import cohere

cohere_api_key = os.getenv("COHERE_API_KEY")
if not cohere_api_key:
    raise ValueError("COHERE_API_KEY not found in environment variables")

co = cohere.Client(cohere_api_key)
logger.info("Cohere client initialized successfully")

[2026-02-10 11:20:08,507] p7606 {2596004477.py:9} INFO - Cohere client initialized successfully


In [58]:
# Retrieve documents for the question
question = "In which year did they build the 100 qubit quantum processor?"
retrieved_docs = retriever.invoke(question)

logger.info(f"Retrieved {len(retrieved_docs)} documents for re-ranking")
for i, doc in enumerate(retrieved_docs):
    logger.info(f"Document {i+1} source: {doc.metadata.get('source', 'unknown')}")
    logger.info(f"Content preview: {doc.page_content[:200]}...")

[2026-02-10 11:20:10,484] p7606 {2766403933.py:5} INFO - Retrieved 4 documents for re-ranking
[2026-02-10 11:20:10,486] p7606 {2766403933.py:7} INFO - Document 1 source: data/2032.txt
[2026-02-10 11:20:10,486] p7606 {2766403933.py:8} INFO - Content preview: Dear Shareholders,

As we conclude another year at QuantumMind, I want to be transparent about the challenges we face while also highlighting the areas where we continue to see potential.

This year h...
[2026-02-10 11:20:10,486] p7606 {2766403933.py:7} INFO - Document 2 source: data/2035.txt
[2026-02-10 11:20:10,486] p7606 {2766403933.py:8} INFO - Content preview: We've gone from the brink of bankruptcy to becoming one of the most valuable and influential tech companies in the world. This turnaround is a testament to the resilience of our team, the power of our...
[2026-02-10 11:20:10,487] p7606 {2766403933.py:7} INFO - Document 3 source: data/2030.txt
[2026-02-10 11:20:10,487] p7606 {2766403933.py:8} INFO - Content preview: This y

In [59]:
# Use Cohere to re-rank the retrieved documents
docs_for_rerank = [doc.page_content for doc in retrieved_docs]

rerank_results = co.rerank(
    model="rerank-english-v3.0",
    query=question,
    documents=docs_for_rerank,
    top_n=len(docs_for_rerank)
)

logger.info("Re-ranking results:")
for result in rerank_results.results:
    logger.info(f"Index: {result.index}, Relevance Score: {result.relevance_score:.4f}")

[2026-02-10 11:20:15,148] p7606 {_client.py:1038} INFO - HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"
[2026-02-10 11:20:15,173] p7606 {897074177.py:11} INFO - Re-ranking results:
[2026-02-10 11:20:15,175] p7606 {897074177.py:13} INFO - Index: 2, Relevance Score: 0.9982
[2026-02-10 11:20:15,176] p7606 {897074177.py:13} INFO - Index: 0, Relevance Score: 0.0084
[2026-02-10 11:20:15,176] p7606 {897074177.py:13} INFO - Index: 3, Relevance Score: 0.0002
[2026-02-10 11:20:15,177] p7606 {897074177.py:13} INFO - Index: 1, Relevance Score: 0.0001


In [60]:
# Get the most relevant document (highest score is first in results)
most_relevant_idx = rerank_results.results[0].index
most_relevant_doc = retrieved_docs[most_relevant_idx]
most_relevant_score = rerank_results.results[0].relevance_score

logger.info(f"Most relevant document index: {most_relevant_idx}")
logger.info(f"Relevance score: {most_relevant_score:.4f}")
logger.info(f"Source: {most_relevant_doc.metadata.get('source', 'unknown')}")
logger.info(f"Content:\n{most_relevant_doc.page_content}")

[2026-02-10 11:20:18,288] p7606 {1683104025.py:6} INFO - Most relevant document index: 2
[2026-02-10 11:20:18,289] p7606 {1683104025.py:7} INFO - Relevance score: 0.9982
[2026-02-10 11:20:18,290] p7606 {1683104025.py:8} INFO - Source: data/2030.txt
[2026-02-10 11:20:18,290] p7606 {1683104025.py:9} INFO - Content:
This year, we've achieved several significant milestones:

We successfully developed our first 100-qubit quantum processor, putting us at the forefront of quantum computing technology. Our AI division has made groundbreaking advancements in quantum machine learning algorithms, showing promise in solving complex optimization problems. We've secured partnerships with leading research institutions and tech giants, positioning us as a key player in the quantum-AI ecosystem. While we're still in the early stages of this technological revolution, the potential applications of our work are vast and transformative. From drug discovery to financial modeling, from climate change mitigat

In [61]:
# Save the most relevant document to task4_answer4.txt
output_content = f"""Question: {question}
Document: {most_relevant_doc.page_content}
"""

with open("task4_answer4.txt", "w") as f:
    f.write(output_content)

logger.info("Saved most relevant document to task4_answer4.txt")

[2026-02-10 11:20:19,856] p7606 {415331199.py:9} INFO - Saved most relevant document to task4_answer4.txt


In [62]:
# Verify the file was created
!cat task4_answer4.txt

Question: In which year did they build the 100 qubit quantum processor?
Document: This year, we've achieved several significant milestones:

We successfully developed our first 100-qubit quantum processor, putting us at the forefront of quantum computing technology. Our AI division has made groundbreaking advancements in quantum machine learning algorithms, showing promise in solving complex optimization problems. We've secured partnerships with leading research institutions and tech giants, positioning us as a key player in the quantum-AI ecosystem. While we're still in the early stages of this technological revolution, the potential applications of our work are vast and transformative. From drug discovery to financial modeling, from climate change mitigation to advanced materials science, QuantumMind is poised to change the world.


## Next steps

Now that we have seen how to use RetrievalQA in action, you can look into advanced topics of:
- Different chunking strategies 
- How to use different Vector stores e.g Amazon OpenSearch, MongoDB etc.
- How to use different search techniques